# Alignment Pipeline Sample Processing Helper

This notebook helps you prepare per-sample configuration YAMLs and execution wrapper scripts for **AlignmentPipeline** (`alignment_pipeline`), supporting all three input modes:
- `"fastq"`: Multi-run / multi-lane paired-end FASTQs via a metadata CSV/TSV sheet
- `"bam"`: Existing BAM files (converted to paired-end FASTQ and realigned)
- `"sra"`: NCBI SRA accessions (downloaded and converted to paired-end FASTQ, with optional `.ngc` key file)

### Guidelines:
1. Copy this notebook into your project workspace directory.
2. Modify the configuration parameters in **Step 2** below (choose `INPUT_MODE` and fill in the matching input section).
3. Run all cells. This notebook will automatically generate per-sample configuration YAMLs and shell runner scripts.
4. In `"fastq"` mode, we expect `PATIENT_ID` and `SAMPLE_TYPE` in `SAMPLE_NAME` to be joined with `_` (e.g., `PID01_TUMOR`).

## Step 1: Import Dependencies
Import standard Python libraries needed for file system operations, YAML handling, and data manipulation.

In [ ]:
import os
import glob
import yaml
import pandas as pd
from pathlib import Path

## Step 2: Define Project Paths and Configuration Parameters
Modify the paths below to point to your data, reference genome, input source, and output directories.

In [ ]:
# ==============================================================================
# 1. Pipeline and Reference Genome Paths
# ==============================================================================
ALIGNMENT_PIPELINE_PATH = "/path/to/alignment_pipeline"  # Absolute path to the cloned pipeline repository
GENOME_FASTA = "/path/to/genome.fa"                      # Reference genome FASTA (must be BWA-indexed)
DBSNP_VCF = "/path/to/dbsnp_all.vcf.gz"                  # Known SNP sites VCF (bgzipped + tabix-indexed, required for BQSR)

# ==============================================================================
# 2. Sequencing Target & Reference Metadata (Required for WES or Panel)
# ==============================================================================
SEQ_TYPE = "WES"                                         # Either "WGS", "WES", or "Panel"
TARGET_FILE = "/path/to/target_regions.bed"              # Absolute path to target regions BED file
BAIT_FILE = "/path/to/bait_regions.bed"                  # Absolute path to bait regions BED file (optional; computed from target if "")
CHROM_SIZES = "/path/to/chrom_sizes.tsv"                 # Chromosome sizes TSV matching reference genome
DICT_GENOME = "/path/to/genome.dict"                     # Reference sequence dictionary (.dict)

# ==============================================================================
# 3. Input Mode & Data Sources (Choose ONE: "fastq", "bam", or "sra")
# ==============================================================================
INPUT_MODE = "fastq"                                     # "fastq", "bam", or "sra"

# Option A (INPUT_MODE == "fastq"): Metadata CSV/TSV describing runs, lanes, reads, fastqs
METADATA_PATH = "/path/to/metadata.csv"

# Option B (INPUT_MODE == "bam") or Option C (INPUT_MODE == "sra"):
# List of sample dicts. For "bam", provide "bamfile". For "sra", provide "sra" (and optional "ngc").
SAMPLES = [
    # {"pid": "PID01", "sample": "TUMOR", "library_prep_kit": "SureSelect_V7", "bamfile": "/path/to/sample.bam"},
    # {"pid": "PID02", "sample": "TUMOR", "library_prep_kit": "SureSelect_V8", "sra": "SRR12345678", "ngc": ""},
]

# ==============================================================================
# 4. Directories for Output Data, Scratch Space, and Generated Scripts
# ==============================================================================
BASE_OUTPUT_PATH = "/path/to/output/directory"           # Where final CRAMs and QC metrics will go
SCRATCH_DIR = ""                                         # Optional scratch dir for intermediate FASTQ/BAMs (defaults to work_dir if "")
SCRIPT_PATH = "/path/to/scripts/directory"               # Directory where sample-specific runner scripts will be saved

# ==============================================================================
# 5. Adapter Trimming & Conversion Parameters
# ==============================================================================
TRIM_ADAPTERS = True                                     # True to run cutadapt + FastQC, False to skip trimming
ADAPTER_FORWARD_READ = ["AGATCGGAAGAGCACACGTCTGAACTCCAGTCA"]  # List of R1 adapter sequences
ADAPTER_REVERSE_READ = ["AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT"]  # List of R2 adapter sequences
ADDITIONAL_CUTADAPT_PARAMETER = ""                       # e.g., "-u 5" for Agilent SureSelect V8 UMI tag trimming
EXTRA_SAMTOOLS_CONVERT_PARAMS = ""                       # Optional extra flags for samtools fastq (BAM mode only)

# ==============================================================================
# 6. Execution / Environment Settings
# ==============================================================================
CONDA_ACTIVATE_PATH = "~/miniforge3/bin/activate"        # Conda activate script path
BASE_ENV_NAME = "halos-base"                             # Name of base Snakemake conda environment
CONDA_PREFIX = ""                                        # Optional --conda-prefix path for shared rule environments
RUN_LOCAL = True                                         # True to run locally with --cores, False for SLURM cluster
LOCAL_CORES = 8

# ==============================================================================
# 7. SLURM Cluster Specific Profiles
# ==============================================================================
SLURM_SMK_PROFILE = "~/smk-profile-v0.1"                 # Profile path (only used if RUN_LOCAL = False)
WORKFLOW_PROFILE = f"{ALIGNMENT_PIPELINE_PATH}/profile"  # Snakemake workflow profile directory
USE_SINGULARITY = True                                   # True to pass --use-singularity
SINGULARITY_PREFIX = ""                                  # Optional --singularity-prefix path
SINGULARITY_ARGS = ""                                    # Optional --singularity-args (e.g. "-B /data:/data")

## Step 3: Load and Preview Input Samples
Load the metadata sheet (in `"fastq"` mode) or preview the `SAMPLES` list (in `"bam"` / `"sra"` mode) before generating configs.

In [ ]:
if INPUT_MODE == "fastq":
    sep = "\t" if METADATA_PATH.split(".")[-1] in ["tsv", "txt"] else ","
    df = pd.read_csv(METADATA_PATH, dtype=str, sep=sep)
    print(f"Loaded metadata sheet with {df.shape[0]} rows and {df.shape[1]} columns.")
    print("Columns present:", list(df.columns))
    display(df.head())
elif INPUT_MODE in ["bam", "sra"]:
    df = pd.DataFrame(SAMPLES)
    print(f"Configured {len(df)} sample(s) for INPUT_MODE='{INPUT_MODE}'.")
    display(df)
else:
    raise ValueError(f"Invalid INPUT_MODE: {INPUT_MODE}. Expected 'fastq', 'bam', or 'sra'.")

## Step 4: Generate Configuration YAMLs and Runner Scripts
For each unique sample, this step will:
1. Create the output workspace directory.
2. Load the base `config/config.yaml` template and clear mutually exclusive input keys.
3. Generate a sample-specific YAML config with the active input source (`metadata`, `bamfile`, or `sra` + optional `ngc`).
4. Write a wrapper `.sh` script to activate the conda environment and launch Snakemake.

In [ ]:
workdir_base = Path(f"{BASE_OUTPUT_PATH}/view-by-pid/")
bash_folder = Path(SCRIPT_PATH)
os.makedirs(bash_folder, exist_ok=True)
os.makedirs(workdir_base, exist_ok=True)

template_config_path = Path("config/config.yaml")
if not template_config_path.exists():
    template_config_path = Path(ALIGNMENT_PIPELINE_PATH) / "config" / "config.yaml"

base_yaml = yaml.safe_load(template_config_path.read_text())

# Build unified list of sample entries to process
sample_entries = []
if INPUT_MODE == "fastq":
    for sample_name in df["SAMPLE_NAME"].unique():
        pid_part, sample_part = sample_name.split("_", 1)
        sample_rows = df[df["SAMPLE_NAME"] == sample_name]
        lib_kits = sample_rows["LIBRARY_PREP_KIT"].dropna().unique()
        if len(lib_kits) > 1:
            raise ValueError(f"Error: Sample {sample_name} has multiple library prep kits: {lib_kits}")
        sample_entries.append({
            "pid": pid_part,
            "sample": sample_part,
            "library_prep_kit": lib_kits[0] if len(lib_kits) == 1 else "Unknown",
            "metadata": METADATA_PATH,
        })
else:
    for entry in SAMPLES:
        sample_entries.append({
            "pid": str(entry["pid"]),
            "sample": str(entry["sample"]),
            "library_prep_kit": entry.get("library_prep_kit", "Unknown"),
            "bamfile": entry.get("bamfile", ""),
            "sra": entry.get("sra", ""),
            "ngc": entry.get("ngc", ""),
        })

count = 0
for entry in sample_entries:
    count += 1
    pid_part = entry["pid"]
    sample_part = entry["sample"]
    sample_name = f"{pid_part}_{sample_part}"
    print(f"[{count}] Processing sample: {sample_name} (mode={INPUT_MODE})")

    yaml_dict = dict(base_yaml)
    for key in ["metadata", "bamfile", "sra", "sra_id", "ngc", "ngc_file", "scratch_dir", "extra_cutadapt_params", "extra_samtools_convert_params"]:
        yaml_dict.pop(key, None)

    sample_workdir = workdir_base / pid_part / sample_part / "merged-alignment"
    sample_logdir = sample_workdir / "logs"
    os.makedirs(sample_workdir, exist_ok=True)

    yaml_dict["work_dir"] = str(sample_workdir)
    yaml_dict["log_dir"] = str(sample_logdir)
    if SCRATCH_DIR:
        yaml_dict["scratch_dir"] = str(Path(SCRATCH_DIR) / sample_name)

    yaml_dict["pid"] = pid_part
    yaml_dict["sample"] = sample_part
    yaml_dict["library_prep_kit"] = entry["library_prep_kit"]
    yaml_dict["SeqType"] = SEQ_TYPE
    yaml_dict["genome"] = GENOME_FASTA
    yaml_dict["dbsnp"] = DBSNP_VCF

    if SEQ_TYPE in ["WES", "Panel"]:
        yaml_dict["target_regions"] = TARGET_FILE
        if BAIT_FILE:
            yaml_dict["bait_regions"] = BAIT_FILE
        else:
            yaml_dict.pop("bait_regions", None)
        yaml_dict["chrom_sizes"] = CHROM_SIZES
        yaml_dict["dict_genome"] = DICT_GENOME

    # Assign mode-specific input
    if INPUT_MODE == "fastq":
        yaml_dict["metadata"] = entry["metadata"]
    elif INPUT_MODE == "bam":
        yaml_dict["bamfile"] = entry["bamfile"]
        if EXTRA_SAMTOOLS_CONVERT_PARAMS:
            yaml_dict["extra_samtools_convert_params"] = EXTRA_SAMTOOLS_CONVERT_PARAMS
    elif INPUT_MODE == "sra":
        yaml_dict["sra"] = entry["sra"]
        if entry.get("ngc"):
            yaml_dict["ngc"] = entry["ngc"]

    # Adapter trimming setup
    yaml_dict["trim_adapters"] = bool(TRIM_ADAPTERS)
    if TRIM_ADAPTERS:
        yaml_dict["adapter_forward_read"] = ADAPTER_FORWARD_READ if isinstance(ADAPTER_FORWARD_READ, list) else [ADAPTER_FORWARD_READ]
        yaml_dict["adapter_reverse_read"] = ADAPTER_REVERSE_READ if isinstance(ADAPTER_REVERSE_READ, list) else [ADAPTER_REVERSE_READ]
        if ADDITIONAL_CUTADAPT_PARAMETER:
            yaml_dict["extra_cutadapt_params"] = ADDITIONAL_CUTADAPT_PARAMETER
    else:
        yaml_dict.pop("adapter_forward_read", None)
        yaml_dict.pop("adapter_reverse_read", None)

    config_file = sample_workdir / f"{sample_name}.yaml"
    with open(config_file, "w") as ff:
        yaml.safe_dump(yaml_dict, ff, sort_keys=False)

    # Build Snakemake CLI flags
    extra_flags = []
    if USE_SINGULARITY:
        extra_flags.append("--use-singularity")
        if SINGULARITY_PREFIX:
            extra_flags.append(f"--singularity-prefix {SINGULARITY_PREFIX}")
        if SINGULARITY_ARGS:
            extra_flags.append(f'--singularity-args "{SINGULARITY_ARGS}"')
    extra_flags.append("--use-conda --conda-frontend mamba")
    if CONDA_PREFIX:
        extra_flags.append(f"--conda-prefix {CONDA_PREFIX}")
    flags_str = " ".join(extra_flags)

    if RUN_LOCAL:
        smk_exec = f"snakemake --cores {LOCAL_CORES} --workflow-profile {WORKFLOW_PROFILE} {flags_str}"
    else:
        smk_exec = f"snakemake --profile {SLURM_SMK_PROFILE} --workflow-profile {WORKFLOW_PROFILE} -j 10 {flags_str}"

    cmd = (
        f"#!/usr/bin/env bash\n"
        f"set -euo pipefail\n"
        f"module purge || true\n"
        f"source {CONDA_ACTIVATE_PATH} {BASE_ENV_NAME}\n"
        f"cd {yaml_dict['work_dir']}\n"
        f"{smk_exec} --configfile {str(config_file)} --snakefile {ALIGNMENT_PIPELINE_PATH}/workflow/Snakefile\n"
    )

    runner_script = bash_folder / f"{count}.sh"
    with open(runner_script, "w") as handle:
        handle.write(cmd)

print(f"\nDone! Generated {count} configuration YAMLs and run scripts in: {bash_folder}")